# Collect: DoD contract awards from USAspending

Pulls Department of Defense contract awards for nine companies, one fiscal year at a time, from the USAspending API (`spending_by_award`). Every response is saved to `data/raw/` before anything else happens to it.

In [1]:
import json
import os
import time

import requests
import pandas as pd

# Every request says who is asking
HEADERS = {"User-Agent": "WebDataScience/1.0 (INFO 4617; brkr6998@colorado.edu)"}

AWARD_URL = "https://api.usaspending.gov/api/v2/search/spending_by_award/"
COUNT_URL = "https://api.usaspending.gov/api/v2/search/spending_by_award_count/"

FIELDS = ["Award ID", "Recipient Name", "Award Amount", "Start Date", "End Date",
          "Awarding Sub Agency", "Description", "NAICS", "PSC",
          "Contract Award Type", "recipient_id"]

# recipient_search_text takes a list, so a company can have several legal names
COMPANIES = {
    "Lockheed Martin": ["Lockheed Martin"],
    "RTX": ["Raytheon", "RTX Corporation", "Pratt & Whitney", "Collins Aerospace", "Rockwell Collins"],
    "Boeing": ["Boeing"],
    "General Dynamics": ["General Dynamics", "Electric Boat", "Bath Iron Works"],
    "Northrop Grumman": ["Northrop Grumman"],
    "Palantir": ["Palantir"],
    "Anduril": ["Anduril"],
    "Shield AI": ["Shield AI"],
    "Scale AI": ["Scale AI"],
}

FISCAL_YEARS = range(2020, 2027)

# 5 pages of 100, sorted by dollars, keeps each company's 500 largest awards per year
MAX_PAGES = 5

os.makedirs("data/raw/pages", exist_ok=True)

In [2]:
def award_filters(company, fiscal_year):
    """The search filters for one company in one fiscal year (October through September)."""
    return {
        "agencies": [{"type": "awarding", "tier": "toptier", "name": "Department of Defense"}],
        "time_period": [{"start_date": f"{fiscal_year - 1}-10-01", "end_date": f"{fiscal_year}-09-30"}],
        "award_type_codes": ["A", "B", "C", "D"],
        "recipient_search_text": COMPANIES[company],
    }


def get_page(company, fiscal_year, page):
    """Return one page of awards, using the saved copy if this page was already pulled."""
    path = f"data/raw/pages/{company.replace(' ', '_')}_FY{fiscal_year}_p{page}.json"
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)

    body = {
        "filters": award_filters(company, fiscal_year),
        "fields": FIELDS,
        "limit": 100,
        "page": page,
        "sort": "Award Amount",
        "order": "desc",
    }
    response = requests.post(AWARD_URL, json=body, headers=HEADERS, timeout=120)
    response.raise_for_status()
    data = response.json()

    with open(path, "w") as f:
        json.dump(data, f)
    time.sleep(1)
    return data


def get_awards(company, fiscal_year):
    """Page through one company's awards for one fiscal year, up to MAX_PAGES pages."""
    rows = []
    for page in range(1, MAX_PAGES + 1):
        data = get_page(company, fiscal_year, page)
        rows.extend(data["results"])
        if not data["page_metadata"]["hasNext"]:
            break
    return rows

Test on one company and year before pulling everything.

In [3]:
test_rows = get_awards("Anduril", 2024)
print(len(test_rows), "awards")
print(json.dumps(test_rows[0], indent=2))

94 awards
{
  "internal_id": 306338225,
  "Award ID": "H9240224F0057",
  "Recipient Name": "ANDURIL INDUSTRIES, INC.",
  "Award Amount": 249978466.39,
  "Start Date": "2024-08-31",
  "End Date": "2027-07-31",
  "Awarding Sub Agency": "U.S. Special Operations Command",
  "Description": "ROADRUNNER, ANVIL AND QUASAR PACKAGES",
  "NAICS": {
    "code": "334511",
    "description": "SEARCH, DETECTION, NAVIGATION, GUIDANCE, AERONAUTICAL, AND NAUTICAL SYSTEM AND INSTRUMENT MANUFACTURING"
  },
  "PSC": {
    "code": "5340",
    "description": "HARDWARE, COMMERCIAL"
  },
  "Contract Award Type": "DELIVERY ORDER",
  "recipient_id": "1869c03a-77bb-b66c-469b-988719bbec4c-C",
  "generated_internal_id": "CONT_AWD_H9240224F0057_9700_H9240222D0001_9700"
}


## How many awards exist

The API stops at 10,000 records per search and the largest companies have far more than that in a year, so this counts what exists to show what the 500-per-year limit leaves out.

In [4]:
def count_awards(company, fiscal_year):
    """How many contract awards the API has for one company in one fiscal year."""
    body = {"filters": award_filters(company, fiscal_year)}
    response = requests.post(COUNT_URL, json=body, headers=HEADERS, timeout=120)
    response.raise_for_status()
    time.sleep(1)
    return response.json()["results"]["contracts"]


if os.path.exists("data/raw/award_counts.csv"):
    counts = pd.read_csv("data/raw/award_counts.csv")
else:
    counts = []
    for company in COMPANIES:
        for fiscal_year in FISCAL_YEARS:
            counts.append({"company": company, "fiscal_year": fiscal_year,
                           "awards_available": count_awards(company, fiscal_year)})
    counts = pd.DataFrame(counts)
    counts.to_csv("data/raw/award_counts.csv", index=False)

print(counts.pivot(index="company", columns="fiscal_year", values="awards_available").to_string())

fiscal_year        2020   2021   2022   2023   2024   2025   2026
company                                                          
Anduril              11     12     29     59     94    106    103
Boeing            24330  21220  20869  21617  20210  18545  11792
General Dynamics   7238   6287   5841   5277   4629   4055   2622
Lockheed Martin   46949  38285  36462  25371   7486   6646   5720
Northrop Grumman   6913   6479   5836   5698   5588   4918   3278
Palantir             12     13      9     16     28     58     48
RTX               12127  11316  10569   9920  13226  10468   4623
Scale AI              1      2      4      2      2      6      4
Shield AI             2      4      5      6      3      3      3


## Pull every company and year

In [5]:
all_rows = []
for company in COMPANIES:
    for fiscal_year in FISCAL_YEARS:
        rows = get_awards(company, fiscal_year)
        for row in rows:
            row["company"] = company
            row["fiscal_year_searched"] = fiscal_year
        all_rows.extend(rows)
        print(f"{company:18} FY{fiscal_year}: {len(rows):>5} awards", flush=True)

with open("data/raw/awards_raw.json", "w") as f:
    json.dump(all_rows, f)

print(len(all_rows), "rows saved")

Lockheed Martin    FY2020:   500 awards


Lockheed Martin    FY2021:   500 awards


Lockheed Martin    FY2022:   500 awards


Lockheed Martin    FY2023:   500 awards


Lockheed Martin    FY2024:   500 awards


Lockheed Martin    FY2025:   500 awards


... (54 outputs left out)

Scale AI           FY2024:     2 awards


Scale AI           FY2025:     6 awards


Scale AI           FY2026:     4 awards


18145 rows saved
